In [11]:
# importing required libraries

import numpy as np
import pandas as pd

In [12]:
# importing messy data in CSV file
df = pd.read_csv(r"C:\Users\Sirmoi Wekesa\Downloads\loan_transactions.csv")
df.head(10)

,loan_id,customer_name,loan_amount,interest_rate,loan_term_months,disbursement_date,status,branch
0,LN001,James Otieno,150000.0,14.5,12,2024-01-15,Active,Nairobi CBD
1,LN002,Wanjiru Kamau,320000.0,12.0,24,2024-02-20,Closed,Westlands
2,LN003,Brian Mutua,NaN,15.5,6,2024-03-10,Active,Mombasa
3,LN004,Fatuma Hassan,9999999.0,11.0,36,2024-01-28,Defaulted,Kisumu
4,LN005,Peter Njoroge,200000.0,13.5,12,2024-04-05,Active,Nairobi CBD
5,LN006,NaN,75000.0,16.0,6,2024-02-14,Closed,Nakuru
6,LN007,Samuel Kipchoge,510000.0,not available,24,2024-03-22,Active,Eldoret
7,LN008,Amina Wekesa,130000.0,NaN,12,2024-05-01,Defaulted,Mombasa
8,LN009,David Mwangi,290000.0,13.0,36,2024-01-30,Closed,Westlands
9,LN010,Rose Adhiambo,60000.0,15.0,6,2024-04-18,Active,Kisumu


In [13]:
# quick look at the data
print(df.dtypes)

loan_id               object
customer_name         object
loan_amount          float64
interest_rate         object
loan_term_months       int64
disbursement_date     object
status                object
branch                object
dtype: object


In [14]:
# Checking for missing/null values per column and print columns with at least 1 missing value
missing = df.isnull().sum()
print(missing[missing > 0])

customer_name    2
loan_amount      4
interest_rate    2
dtype: int64


In [15]:
# Checking for duplicate load IDs. we assume that loan id is the unique identifier
duplicates = df[df.duplicated(subset = "loan_id", keep = False)]
print(duplicates[['loan_id','customer_name','loan_amount']])

   loan_id    customer_name  loan_amount
0    LN001     James Otieno     150000.0
10   LN001     James Otieno     150000.0
16   LN016  Beatrice Atieno     220000.0
21   LN016  Beatrice Atieno     220000.0


In [16]:
#Checking for wrong data types in interest rate column.
# Main aim is find any non-numeric values in the interest_rate column that could cause errors in calculations.
# creating a function called is_not_numeric that checks if a value/val is not a number. 
def is_not_numeric(val):
# try, attempts to convert the value into a float. If it works, the value is numeric, so the function returns False (not bad)
    try:
        float(val)
        return False
# If conversion fails, a ValueError or TypeError occurs.The function returns True, marking the value as non-numeric.
    except (ValueError, TypeError):
        return True
# below, i run the function on every row in the interest rate column and i filters the dataframe to only rows where is_not_numeric is True
bad_types = df[df['interest_rate'].apply(is_not_numeric)]
print(bad_types[['loan_id','customer_name','interest_rate']])

   loan_id    customer_name  interest_rate
6    LN007  Samuel Kipchoge  not available
14   LN014   Lilian Muthoni  not available
25   LN024      Diana Njeri  not available


In [17]:
# Checking for outliers in the loan amount column
# First converting loan_amount to numeric since some rows have blanks.

df['loan_amount'] = pd.to_numeric(df['loan_amount'], errors = 'coerce')

# Finding the mean and std

mean = df['loan_amount'].mean()
std = df['loan_amount'].std()

# Then flagging any amount more than 3 standard deviations above the mean.
outliers = df[df['loan_amount'] > mean + 3 * std]

print(outliers[["loan_id", "customer_name", "loan_amount"]])


Empty DataFrame
Columns: [loan_id, customer_name, loan_amount]
Index: []


In [18]:
# Checking for inconsistent status values
print(df['status'].value_counts())

status
Active       15
Closed        8
Defaulted     5
ACTIVE        1
active        1
CLOSED        1
DEFAULTED     1
Name: count, dtype: int64


In [19]:
# We can also chacke the branch names. You'll spot "NBI CBD" hiding among the proper branch names.
print(df['branch'].value_counts())

branch
Nairobi CBD    7
Kisumu         5
Westlands      4
Mombasa        4
Nakuru         4
Eldoret        3
Garissa        2
Thika          2
NBI CBD        1
Name: count, dtype: int64


In [23]:
# DATA QUALITY CHECK AND SCORING 

# Total rows in the dataset
total_rows = len(df)

# Count duplicates (assumes 'loan_id' is the unique identifier)
duplicate_count = int(df['loan_id'].duplicated().sum())

# Count missing values across the entire DataFrame
missing_count = int(df.isna().sum().sum())

# Count wrong data types (from previous bad_types check)
bad_types_count = int(len(bad_types))

# Count outliers (from previous outliers check)
outlier_count = int(len(outliers))

# Count status casing issues
casing_issues = int((df["status"] != df["status"].str.title()).sum())

# Count branch name issues
branch_issues = int((df["branch"] == "NBI CBD").sum())

# Total issues
total_issues = (missing_count + duplicate_count + bad_types_count +
                outlier_count + casing_issues + branch_issues)

# Data quality score calculation
quality_score = round(((total_rows - total_issues) / total_rows) * 100, 1)

# Print detailed counts and score
print(f"Total Rows Analysed  : {total_rows}")
print(f"Missing Values       : {missing_count}")
print(f"Duplicate IDs        : {duplicate_count}")
print(f"Wrong Data Types     : {bad_types_count}")
print(f"Outliers Detected    : {outlier_count}")
print(f"Status Casing Issues : {casing_issues}")
print(f"Branch Name Issues   : {branch_issues}")
print(f"DATA QUALITY SCORE   : {quality_score}%")

# Interpret the quality score
if quality_score >= 90:
    print('Status: Good, Minor Clean-up Needed')
elif quality_score >= 70:
    print('Status: Fair, Significant Issues found')
else:
    print('Status: Poor, Major Cleaning Required')

Total Rows Analysed  : 32
Missing Values       : 8
Duplicate IDs        : 2
Wrong Data Types     : 3
Outliers Detected    : 0
Status Casing Issues : 4
Branch Name Issues   : 1
DATA QUALITY SCORE   : 43.8%
Status: Poor, Major Cleaning Required
